In [1]:
# Clone repos + install
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Download Pile validation set (needed if re-generating scales)
!mkdir -p smoothquant_repo/dataset
!wget -q -O smoothquant_repo/dataset/val.jsonl.zst \
    https://huggingface.co/datasets/mit-han-lab/pile-val-backup/resolve/main/val.jsonl.zst

# Copy activation scales from Drive
!mkdir -p smoothquant_repo/act_scales
!cp /content/drive/MyDrive/thesis_results/act_scales/*.pt smoothquant_repo/act_scales/

# Verify
!nvidia-smi
!ls -la smoothquant_repo/act_scales/
!python -c "from smoothquant.smooth import smooth_lm; print('smoothquant OK')"

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 97 (delta 29), reused 87 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 4.81 MiB | 22.38 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 23.68 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
Mounted at /content/drive
Mon May  4 08:07:17 2026       
+---------------------------------------------------

In [2]:
%%writefile /content/llm-quantization-thesis/run_scheme_compare.py
"""
Parameterized scheme comparison script.
Based on smoothquant/ppl_eval.py but accepts --weight_quant and --act_quant args.
"""
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from smoothquant.smooth import smooth_lm
from smoothquant.fake_quant import quantize_model
from datasets import load_dataset
import argparse
import json
import os
import time
import tqdm


class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=40):
        self.dataset = tokenizer(
            "\n\n".join(dataset["text"]), return_tensors="pt"
        ).input_ids.to(device)
        self.n_samples = n_samples

    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        nlls = []
        n_samples = self.n_samples if self.n_samples else self.dataset.size(1) // 2048
        for i in tqdm.tqdm(range(n_samples), desc="Evaluating"):
            batch = self.dataset[:, (i * 2048) : ((i + 1) * 2048)].to(model.device)
            with torch.no_grad():
                lm_logits = model(batch).logits
            shift_logits = lm_logits[:, :-1, :].contiguous().float()
            shift_labels = self.dataset[:, (i * 2048) : ((i + 1) * 2048)][:, 1:]
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)
            )
            neg_log_likelihood = loss.float() * 2048
            nlls.append(neg_log_likelihood)
        return torch.exp(torch.stack(nlls).sum() / (n_samples * 2048))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_path", type=str, required=True)
    parser.add_argument("--act_scales_path", type=str, required=True)
    parser.add_argument("--alpha", type=float, default=0.5)
    parser.add_argument("--smooth", action="store_true")
    parser.add_argument("--quantize", action="store_true")
    parser.add_argument("--weight_quant", type=str, default="per_channel",
                        choices=["per_channel", "per_tensor"])
    parser.add_argument("--act_quant", type=str, default="per_token",
                        choices=["per_token", "per_tensor"])
    parser.add_argument("--quantize_bmm", action="store_true", default=True)
    parser.add_argument("--config_label", type=str, default="unknown",
                        help="Label for this config (e.g. O1, O2, C, D)")
    parser.add_argument("--save_json", type=str, default=None,
                        help="Path to save result JSON")
    args = parser.parse_args()

    start = time.time()

    print("=" * 60)
    print(f"  Config: {args.config_label}")
    print(f"  Model:  {args.model_path}")
    print(f"  Smooth: {args.smooth} (alpha={args.alpha})")
    print(f"  Quant:  {args.quantize}")
    if args.quantize:
        print(f"  Weight: {args.weight_quant}")
        print(f"  Act:    {args.act_quant}")
        print(f"  BMM:    {args.quantize_bmm}")
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(args.model_path)
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    evaluator = Evaluator(dataset, tokenizer, "cuda")

    model = AutoModelForCausalLM.from_pretrained(
        args.model_path, torch_dtype=torch.bfloat16, device_map="auto"
    )

    if args.smooth:
        act_scales = torch.load(args.act_scales_path)
        smooth_lm(model, act_scales, args.alpha)
        print("Smoothing applied.")

    if args.quantize:
        model = quantize_model(
            model,
            weight_quant=args.weight_quant,
            act_quant=args.act_quant,
            quantize_bmm_input=args.quantize_bmm,
        )
        print("Quantization applied.")

    ppl = evaluator.evaluate(model)
    elapsed = time.time() - start
    ppl_val = ppl.item()

    print(f"\n>>> Config {args.config_label}: Perplexity = {ppl_val:.4f} ({elapsed:.0f}s)")

    # Save result
    result = {
        "config_label": args.config_label,
        "model": args.model_path,
        "smooth": args.smooth,
        "alpha": args.alpha if args.smooth else None,
        "quantize": args.quantize,
        "weight_quant": args.weight_quant if args.quantize else None,
        "act_quant": args.act_quant if args.quantize else None,
        "quantize_bmm": args.quantize_bmm if args.quantize else None,
        "wikitext2_ppl": round(ppl_val, 4),
        "duration_seconds": round(elapsed, 1),
    }

    if args.save_json:
        os.makedirs(os.path.dirname(args.save_json), exist_ok=True)
        with open(args.save_json, "w") as f:
            json.dump(result, f, indent=2)
        print(f"Saved to {args.save_json}")

    return result


if __name__ == "__main__":
    main()

Writing /content/llm-quantization-thesis/run_scheme_compare.py


In [3]:
%cd /content/llm-quantization-thesis

!python run_scheme_compare.py \
    --model_path facebook/opt-6.7b \
    --act_scales_path smoothquant_repo/act_scales/opt-6.7b.pt \
    --config_label FP16 \
    --save_json results/task01/opt-6.7b_FP16.json

/content/llm-quantization-thesis
  Config: FP16
  Model:  facebook/opt-6.7b
  Smooth: False (alpha=0.5)
  Quant:  False
config.json: 100% 651/651 [00:00<00:00, 3.75MB/s]
tokenizer_config.json: 100% 685/685 [00:00<00:00, 4.58MB/s]
vocab.json: 899kB [00:00, 106MB/s]
merges.txt: 456kB [00:00, 94.6MB/s]
special_tokens_map.json: 100% 441/441 [00:00<00:00, 3.33MB/s]
README.md: 10.5kB [00:00, 37.6MB/s]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): 100% 733k/733k [00:01<00:00, 513kB/s] 
wikitext-2-raw-v1/train-00000-of-00001.p(…): 100% 6.36M/6.36M [00:00<00:00, 15.5MB/s]
wikitext-2-raw-v1/validation-00000-of-00(…): 100% 657k/657k [00:00<00:00, 1.08MB/s]
Generating test split: 100% 4358/4358 [00:00<00:00, 131102.08 examples/s]
Generating train split: 100% 36718/36718 [00:00<00:00, 812909.17 examples/s]
Generating validation split: 100% 3760/3760 [00:00<00:00, 674388.84 examples/s]
`torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin.index.json: 41.9kB [00:00, 101MB/s]

model.safet

In [4]:
!python run_scheme_compare.py \
    --model_path facebook/opt-6.7b \
    --act_scales_path smoothquant_repo/act_scales/opt-6.7b.pt \
    --quantize \
    --weight_quant per_tensor --act_quant per_tensor \
    --config_label W8A8-naive \
    --save_json results/task01/opt-6.7b_W8A8-naive.json

  Config: W8A8-naive
  Model:  facebook/opt-6.7b
  Smooth: False (alpha=0.5)
  Quant:  True
  Weight: per_tensor
  Act:    per_tensor
  BMM:    True
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 516/516 [00:04<00:00, 110.27it/s, Materializing param=model.decoder.layers.31.self_attn_layer_norm.weight]
Quantization applied.
Evaluating: 100% 40/40 [00:26<00:00,  1.52it/s]

>>> Config W8A8-naive: Perplexity = 26.0056 (292s)
Saved to results/task01/opt-6.7b_W8A8-naive.json


In [5]:
!python run_scheme_compare.py \
    --model_path facebook/opt-6.7b \
    --act_scales_path smoothquant_repo/act_scales/opt-6.7b.pt \
    --smooth --alpha 0.5 --quantize \
    --weight_quant per_tensor --act_quant per_token \
    --config_label SQ-O1 \
    --save_json results/task01/opt-6.7b_SQ-O1.json

  Config: SQ-O1
  Model:  facebook/opt-6.7b
  Smooth: True (alpha=0.5)
  Quant:  True
  Weight: per_tensor
  Act:    per_token
  BMM:    True
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 516/516 [00:04<00:00, 106.90it/s, Materializing param=model.decoder.layers.31.self_attn_layer_norm.weight]
Smoothing applied.
Quantization applied.
Evaluating: 100% 40/40 [00:26<00:00,  1.52it/s]

>>> Config SQ-O1: Perplexity = 10.7041 (292s)
Saved to results/task01/opt-6.7b_SQ-O1.json


In [6]:
!python run_scheme_compare.py \
    --model_path facebook/opt-6.7b \
    --act_scales_path smoothquant_repo/act_scales/opt-6.7b.pt \
    --smooth --alpha 0.5 --quantize \
    --weight_quant per_tensor --act_quant per_tensor \
    --config_label SQ-O2 \
    --save_json results/task01/opt-6.7b_SQ-O2.json

  Config: SQ-O2
  Model:  facebook/opt-6.7b
  Smooth: True (alpha=0.5)
  Quant:  True
  Weight: per_tensor
  Act:    per_tensor
  BMM:    True
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 516/516 [00:04<00:00, 109.67it/s, Materializing param=model.decoder.layers.31.self_attn_layer_norm.weight]
Smoothing applied.
Quantization applied.
Evaluating: 100% 40/40 [00:26<00:00,  1.52it/s]

>>> Config SQ-O2: Perplexity = 10.7165 (294s)
Saved to results/task01/opt-6.7b_SQ-O2.json


In [7]:
!python run_scheme_compare.py \
    --model_path facebook/opt-6.7b \
    --act_scales_path smoothquant_repo/act_scales/opt-6.7b.pt \
    --smooth --alpha 0.5 --quantize \
    --weight_quant per_channel --act_quant per_token \
    --config_label SQ-PCW-PT \
    --save_json results/task01/opt-6.7b_SQ-PCW-PT.json

  Config: SQ-PCW-PT
  Model:  facebook/opt-6.7b
  Smooth: True (alpha=0.5)
  Quant:  True
  Weight: per_channel
  Act:    per_token
  BMM:    True
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 516/516 [00:04<00:00, 111.89it/s, Materializing param=model.decoder.layers.31.self_attn_layer_norm.weight]
Smoothing applied.
Quantization applied.
Evaluating: 100% 40/40 [00:26<00:00,  1.51it/s]

>>> Config SQ-PCW-PT: Perplexity = 10.7210 (292s)
Saved to results/task01/opt-6.7b_SQ-PCW-PT.json


In [8]:
!python run_scheme_compare.py \
    --model_path facebook/opt-6.7b \
    --act_scales_path smoothquant_repo/act_scales/opt-6.7b.pt \
    --smooth --alpha 0.5 --quantize \
    --weight_quant per_channel --act_quant per_tensor \
    --config_label SQ-PCW-TEN \
    --save_json results/task01/opt-6.7b_SQ-PCW-TEN.json

  Config: SQ-PCW-TEN
  Model:  facebook/opt-6.7b
  Smooth: True (alpha=0.5)
  Quant:  True
  Weight: per_channel
  Act:    per_tensor
  BMM:    True
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 516/516 [00:04<00:00, 110.76it/s, Materializing param=model.decoder.layers.31.self_attn_layer_norm.weight]
Smoothing applied.
Quantization applied.
Evaluating: 100% 40/40 [00:26<00:00,  1.52it/s]

>>> Config SQ-PCW-TEN: Perplexity = 10.7294 (293s)
Saved to results/task01/opt-6.7b_SQ-PCW-TEN.json


In [9]:
!mkdir -p /content/drive/MyDrive/thesis_results/task01
!cp results/task01/*.json /content/drive/MyDrive/thesis_results/task01/

# Print summary
import json, glob
print(f"\n{'Config':<15} {'PPL':>10} {'Time':>8}")
print("-" * 35)
for f in sorted(glob.glob("results/task01/opt-6.7b_*.json")):
    r = json.load(open(f))
    print(f"{r['config_label']:<15} {r['wikitext2_ppl']:>10.4f} {r['duration_seconds']:>7.1f}s")


Config                 PPL     Time
-----------------------------------
FP16               10.6714    96.7s
SQ-O1              10.7041   292.2s
SQ-O2              10.7165   294.4s
SQ-PCW-PT          10.7210   292.2s
SQ-PCW-TEN         10.7294   293.0s
W8A8-naive         26.0056   292.5s
